In [21]:
import pickle
import librosa
import numpy as np
import IPython.display as ipd
from scipy.signal import stft, istft
from sklearn.model_selection import train_test_split

In [22]:
with open("dataset/dataset.pkl", "rb") as f:
    dataset = pickle.load(f)
    x_files = dataset["gtr"]
    y_files = dataset["ney"]

x_train_files, x_test_files, y_train_files, y_test_files = train_test_split(
    x_files, y_files, test_size=0.2, random_state=42)

print(len(x_train_files), len(x_test_files))

200 50


In [23]:
NEY_SPECTROGRAM_DIR = "dataset/spectrograms/ney/"
GTR_SPECTROGRAM_DIR = "dataset/spectrograms/ac_gtr/"

In [24]:
spectrogram = np.array([
    np.load(GTR_SPECTROGRAM_DIR + x_train_files[0], allow_pickle=True),
    np.load(GTR_SPECTROGRAM_DIR + x_train_files[1], allow_pickle=True)
])

In [25]:
spectrogram = np.expand_dims(spectrogram, axis=3)
spectrogram.shape

(2, 256, 40, 1)

In [26]:
spec_tr = np.transpose(spectrogram, (0, 3, 2, 1))
spec_tr.shape

(2, 1, 40, 256)

spectrogram -> audio

In [27]:
SR = 48000
WINDOW_LENGTH = 10200
FRAME_SIZE = 1024
HOP_LENGTH = 256

In [28]:
signal, _ = librosa.load("dataset/ney/00_Ney_C_3.wav", mono=True, sr=SR)
ipd.Audio(signal, rate=SR)

In [ ]:
stft(signal, fs=SR)

In [17]:
signal, _ = librosa.load("dataset/ney/00_Ney_C_3.wav", mono=True, sr=SR)
num_windows = int(len(signal) / WINDOW_LENGTH)
padded_inverse_list = []
for i in range(num_windows):
    signal_window = signal[i*WINDOW_LENGTH: (i+1) * WINDOW_LENGTH]
    print(f"SW Len: {len(signal_window)}")
    y_pad = librosa.util.fix_length(
        signal_window, size=len(signal_window) + FRAME_SIZE // 2)
    print(f"Y Pad Len: {len(y_pad)}")

    padded_spec = librosa.stft(
        y_pad, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH)[:-1]
    print(f"padded_spec Len: {len(padded_spec)}")
    padded_spec = np.abs(padded_spec)
    log_spec = librosa.amplitude_to_db(padded_spec)
    print(f"log_spec Len: {len(log_spec)}")

    log_spec_inv = librosa.db_to_amplitude(log_spec)
    padded_inv = librosa.istft(
        log_spec_inv, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH, length=10200)
    print(f"padded_inv Len: {len(padded_inv)}")
    padded_inverse_list.extend(padded_inv)
    print("-" * 50)

print(len(signal), len(padded_inverse_list))

SW Len: 10200
Y Pad Len: 10712
padded_spec Len: 512
log_spec Len: 512
padded_inv Len: 10200
--------------------------------------------------
SW Len: 10200
Y Pad Len: 10712
padded_spec Len: 512
log_spec Len: 512
padded_inv Len: 10200
--------------------------------------------------
SW Len: 10200
Y Pad Len: 10712
padded_spec Len: 512
log_spec Len: 512
padded_inv Len: 10200
--------------------------------------------------
SW Len: 10200
Y Pad Len: 10712
padded_spec Len: 512
log_spec Len: 512
padded_inv Len: 10200
--------------------------------------------------
SW Len: 10200
Y Pad Len: 10712
padded_spec Len: 512
log_spec Len: 512
padded_inv Len: 10200
--------------------------------------------------
SW Len: 10200
Y Pad Len: 10712
padded_spec Len: 512
log_spec Len: 512
padded_inv Len: 10200
--------------------------------------------------
SW Len: 10200
Y Pad Len: 10712
padded_spec Len: 512
log_spec Len: 512
padded_inv Len: 10200
--------------------------------------------------

In [18]:
ipd.Audio(signal, rate=SR)

In [19]:
ipd.Audio(padded_inverse_list, rate=SR)